# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Lane Chosen:** Lane 2 — Refresh / Content Opportunity Scoring

**ML Task Type:** **Scoring / Ranking** (built on a binary classification foundation)

**Why Ranking / Scoring?**
Our primary objective is decision-support for a content review team with limited weekly review capacity. Rather than deciding yes/no on every page in isolation (which classification does), we need to sort the entire content inventory from highest to lowest risk/opportunity. By ranking the pages, editors can start from the top of the queue and work down as far as their capacity allows. Under the hood, this is implemented by training a binary classifier to output a probability of traffic decline, which serves as a score to rank the content queue.

In [1]:
# Programmatic task classification confirmation
print("Task Type: Ranking / Priority Scoring (derived from binary classification probabilities)")

Task Type: Ranking / Priority Scoring (derived from binary classification probabilities)


## 2. Target or proxy

**Target Concept:** Real-world traffic decline or performance loss.

**Target Variable in Starter Data:** `is_declining_label` (defined as `trend_direction == 'down'`).

**Observed Outcome vs. Rule-based Label:**
* **Observed Outcome:** The label `trend_direction` is calculated directly from measured search performance (a comparison of recent traffic trends over a rolling window). It is not a rule defined by the FlyRank application's choices or flags (like `health_score` or `priority_score`).
* **Proxy Label Limitation:** While `is_declining_label` is observed, it is computed from the same window as the starter features. In a production system (like the full warehouse release), we would ideally construct a **future-window label** (e.g. features from days 0-90, predicting a decline measured in days 91-120) to prevent data leakage and capture future outcomes.

In [2]:
# Target definition sketch
import pandas as pd
# Target expression in our pipeline:
# is_declining_label = (df['trend_direction'] == 'down')
print("Target Label: is_declining_label (Boolean)")

Target Label: is_declining_label (Boolean)


## 3. Success metric

**Success Metric:** **Precision@50** (and **Average Precision**)

**Why is this metric defensible?**
* **Capacity Alignment:** In a real-world scenario, editors have a finite budget (e.g., they can review and update 50 pages per week).
* **Minimizing Wasted Effort:** A high Precision@50 guarantees that of the top 50 pages recommended in our queue, a high percentage are actually in decline. If precision is low (e.g., 20%), editors waste 80% of their time reviewing stable or growing pages.
* **Average Precision (AP):** We will also evaluate AP to ensure the model ranks true positives higher up in the queue overall.

In [3]:
# Success metric targets
print("Primary Metric: Precision@50")
print("Secondary Metric: Average Precision (AP)")

Primary Metric: Precision@50
Secondary Metric: Average Precision (AP)


## 4. The unit of analysis, as a real dataframe

**Unit of Analysis:** **One row = One unique content item (page)** (uniquely identified by `content_id` for a given `client_id`).

We load the starter slice of the dataset below. To ensure the quality of our unit of analysis, we filter out noise by keeping only pages with at least one impression (`impressions_90d > 0`) and pages that are at least 90 days old (`content_age_days >= 90`).

In [4]:
import pandas as pd

# Load the starter slice
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

# Filter out noise (pages with no search visibility or pages that are too new)
df_slice = df[(df['impressions_90d'] > 0) & (df['content_age_days'] >= 90)].copy()

# Define our target column
df_slice['is_declining_label'] = (df_slice['trend_direction'] == 'down').astype(int)

# Display shape and unit of analysis dataframe
print(f"Dataframe slice shape: {df_slice.shape}")
print(f"Unit of analysis: unique page per client.")

# Show key columns demonstrating unit of analysis and target
cols_to_show = ['content_id', 'client_id', 'impressions_90d', 'content_age_days', 'days_since_last_update', 'avg_position', 'is_declining_label']
print("\nDataframe Sample:")
display(df_slice[cols_to_show].head(5))

Dataframe slice shape: (30000, 45)
Unit of analysis: unique page per client.

Dataframe Sample:


,content_id,client_id,impressions_90d,content_age_days,days_since_last_update,avg_position,is_declining_label
0,content_304f48230142,client_f369cb89fc,3803,187,20,10.6,1
1,content_a1fb4e703a9e,client_4e07408562,15320,445,25,20.3,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,141,20,36.5,1
3,content_331d6c4de07b,client_19581e27de,11751,463,22,6.2,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,263,14,44.0,1


## 5. Why ML beats a fixed rule here

A fixed, heuristic-based rule (e.g., "refresh a page if it is older than 180 days and has declining clicks") is insufficient for several key reasons:

1. **Tangled Multi-dimensional Signals:** Traffic decline is not determined by age alone. It depends on a combination of keyword search volume, CPC, competitiveness, actual position, position decay rates, and on-page user engagement (scroll rate, GA4 session length).
2. **Non-linear Interactions:** A high position decay is critical for high-volume pages, but negligible for pages with zero demand. Heuristics struggle to capture these non-linear interactions without becoming a massive, unmaintainable tree of nested if-statements.
3. **Client-level Variance:** Different clients have vastly different traffic and engagement baselines. A fixed threshold (e.g. CTR < 1%) might represent excellent performance for one page format but a complete failure for another. Machine learning models can learn these relationships dynamically from the data rather than relying on arbitrary, hand-tuned constants.

In [5]:
# Demonstrate complexity (e.g. overlap of multiple conditions)
# Show that a simple age-based baseline has low correlation with actual decline
age_corr = df_slice['content_age_days'].corr(df_slice['is_declining_label'])
print(f"Correlation between page age and traffic decline label: {age_corr:.3f}")
print("The correlation is extremely weak, proving that simple rules based on age alone cannot accurately predict decline.")

Correlation between page age and traffic decline label: -0.164
The correlation is extremely weak, proving that simple rules based on age alone cannot accurately predict decline.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.